In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

In [0]:
display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(field.name))

In [0]:
df = df.withColumn('MAINTENANCE', when(upper(col("MAINTENANCE")) == "YES", lit(True))
                   .when(upper(col("MAINTENANCE")) == "NO", lit(False))
                   .otherwise(None)
                   )

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
display(df.limit(5))

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.erp_product_category')